# 10. Оценка через EvaluationOrchestrator (с известной истиной)

**Цель:** показать `EvaluationCase`/`EvaluationOrchestrator` на одном многоканальном случае с полной известной истиной: время события и исходный канал. В отличие от неразмеченного случая (`has_truth=False`), `kuramoto_propagation_series.csv` несёт синтетическую разметку -- здесь оркестратор считает не только hit/miss, но и ранг истинного источника (Top-1/Top-k/MRR).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, EvaluationCase, EvaluationOrchestrator, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_propagation_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_propagation_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df.columns if column.startswith("channel_")]
x = df[channel_names].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 2

propagation_metadata = json.loads((ROOT / "data/generated/synthetic_datasets_metadata.json").read_text())["kuramoto_propagation"]
event_onset = int(np.flatnonzero(event_mask)[0])
source_channel = int(propagation_metadata["source_channel"])
channel_names[source_channel]

'channel_00'

In [2]:
BASELINE_SIZE = event_onset  # oracle-assisted: valid only because this synthetic onset is known
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.85, alarm_probability=0.97, top_k=3, persistence=1),
    graph=GraphConfig(method="ar", edge_threshold=0.35, random_state=SEED),
    input_config=InputConfig(missing="error", require_regular_time=True),
    verbose=True,
)
evaluation = EvaluationOrchestrator(pipeline, early_tolerance=0, k=3, verbose=True).evaluate([
    EvaluationCase(x, event_time=event_onset, source=source_channel, case_id="Kuramoto propagation"),
])

print("Summary:", evaluation.summary())
pd.DataFrame(evaluation.to_records())

Evaluating cases:   0%|          | 0/1 [00:00<?, ?it/s]

Summary: {'n-d': {'cases': 1, 'hit': 1, 'miss': 0, 'early_alarm': 0, 'false_alarm': 0, 'correct_rejection': 0, 'unlabelled': 0, 'detection_rate': 1.0, 'false_alarm_rate': nan, 'mean_delay': 17.0, 'median_delay': 17.0, 'heuristic_n': 1, 'heuristic_top1': 1.0, 'heuristic_top3': 1.0, 'heuristic_mrr': 1.0, 'gnn_n': 0, 'gnn_top1': nan, 'gnn_top3': nan, 'gnn_mrr': nan, 'gat_n': 0, 'gat_top1': nan, 'gat_top3': nan, 'gat_mrr': nan}}


,case_id,route,outcome,stop_reason,detected_time,true_event_time,delay,true_source,heuristic_source,heuristic_rank,gnn_source,gnn_rank,gat_source,gat_rank
0,Kuramoto propagation,n-d,hit,None,949,932,17,0,0,1,None,None,None,None


`heuristic_rank=1` означает, что среди всех 12 каналов истинный источник получил наибольший вес прозрачного ranking -- не приближение, а точный Top-1. `gnn_rank`/`gat_rank` здесь пусты, потому что этому pipeline не передан обученный `process_model` (обучение -- в ноутбуке 9); без него оркестратор честно не сообщает GNN/GAT-метрики, а не подставляет эвристику вместо них.

Один случай по-прежнему не образует benchmark: `heuristic_top1=1.0` в `summary()` -- это `1/1`, а не статистика по независимой выборке.

**Самопроверка:** почему `early_tolerance=0` здесь оправдан, а в ноутбуке 9 использован `early_tolerance=25`? Что покажет `evaluation.summary()`, если добавить в батч второй случай с намеренно неверным `source`? Почему `heuristic_rank=1` на этом ряде не означает, что эвристика вообще не ошибается (см. ноутбук 5)?